# Lab 9-10: Image Processing


In [ ]:
# Jupyter/local: install once (Colab already has these). Restart the kernel afterwards if asked.
# %pip install opencv-python numpy matplotlib scikit-image scikit-learn requests
import os, csv, heapq, itertools, cv2, numpy as np, matplotlib.pyplot as plt
from collections import Counter
IMG_DIR, OUT_DIR = "images", "outputs"
os.makedirs(IMG_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

def load(name, gray=False, max_side=None):
    path = os.path.join(IMG_DIR, name)
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE if gray else cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Cannot read '{path}'. Run the image upload cell first.")
    if max_side and max(img.shape[:2]) > max_side:
        s = max_side / max(img.shape[:2])
        img = cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    return img

def show(ax, img, title, cmap="gray"):
    if img.ndim == 3: ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    else: ax.imshow(img, cmap=cmap)
    ax.set_title(title, fontsize=9); ax.axis("off")

def save(fig, name):
    p = os.path.join(OUT_DIR, name); fig.savefig(p, dpi=150, bbox_inches="tight"); print("saved", p)

## Automatic image download (Wikimedia Commons)
This cell searches Wikimedia Commons, downloads real photographs, and saves them with the exact filenames the lab needs. It also writes `images/sources.csv` with each photo's page, direct link, licence and author for your report. It keeps only JPEG photos uploaded before 2022 with a stated licence, and skips anything mentioning AI or generated images, to avoid AI-generated pictures. If a picture looks unsuitable in the preview, set `PICK["file.jpg"] = 1` (or 2, 3...) in the cell below and run it again to get the next candidate.

In [ ]:
import requests, re, time, csv
API = "https://commons.wikimedia.org/w/api.php"
HDR = {"User-Agent": "LabAssignmentImageFetcher/1.0 (student image-processing lab)"}

PICK = {n: 1 for n in ['outdoor.jpg','gray_photo.jpg','compress_photo.jpg','jpeg_image.jpg','textures.jpg']}   # friend version uses 2nd match; example: PICK["leaf.jpg"] = 1  -> use the 2nd candidate instead of the 1st

TARGETS = {
    "outdoor.jpg":          ["village road houses trees", "harbour town buildings road", "mountain town street buildings"],
    "gray_photo.jpg":       ["church tower photograph", "temple architecture photograph", "palace facade photograph"],
    "compress_photo.jpg":   ["forest path photograph", "river valley landscape photograph", "meadow flowers landscape photograph"],
    "compress_document.png":["manuscript page text", "old book page typed text", "printed text document scan"],
    "jpeg_image.jpg":       ["windmill photograph", "market square photograph", "railway station photograph"],
    "textures.jpg":         ["brick wall and grass photograph", "stone wall and wooden door photograph", "roof tiles and wooden fence photograph"],
    "leaf.jpg":             ["leaf rust disease", "leaf blight spots tomato", "powdery mildew leaf"],
}
AI_WORDS = re.compile(r"\bAI\b|artificial intelligence|generated|midjourney|stable diffusion|dall-?e", re.I)
clean = lambda h: re.sub(r"<[^>]+>", "", h or "").strip()

def search(query, limit=40):
    p = dict(action="query", format="json", generator="search", gsrsearch=query + " filetype:bitmap",
             gsrnamespace=6, gsrlimit=limit, prop="imageinfo",
             iiprop="url|size|mime|timestamp|extmetadata", iiurlwidth=1200)
    for _ in range(4):
        r = requests.get(API, params=p, headers=HDR, timeout=60)
        if r.status_code == 429: time.sleep(5); continue
        r.raise_for_status(); break
    pages = r.json().get("query", {}).get("pages", {})
    found = []
    for pg in sorted(pages.values(), key=lambda x: x.get("index", 0)):
        ii = (pg.get("imageinfo") or [{}])[0]
        meta = ii.get("extmetadata", {})
        lic = clean(meta.get("LicenseShortName", {}).get("value"))
        text = " ".join(clean(meta.get(k, {}).get("value")) for k in ("ImageDescription", "Categories", "ObjectName"))
        if (ii.get("mime") != "image/jpeg" or ii.get("width", 0) < 1000 or not lic
                or ii.get("timestamp", "9999") >= "2022-01-01" or AI_WORDS.search(text + pg.get("title", ""))):
            continue
        found.append(dict(title=pg["title"], page=ii.get("descriptionurl"), direct=ii.get("url"),
                          dl=ii.get("thumburl") or ii.get("url"), license=lic,
                          author=clean(meta.get("Artist", {}).get("value"))[:80],
                          desc=clean(meta.get("ImageDescription", {}).get("value"))[:150],
                          uploaded=ii.get("timestamp", "")[:10]))
    return found

records, used = [], set()
for fname, queries in TARGETS.items():
    cands = []
    for q in queries:
        try: cands += [c for c in search(q) if c["page"] not in {x["page"] for x in cands}]
        except Exception as e: print("search failed:", q, e)
        time.sleep(1)
    cands = [c for c in cands if c["page"] not in used]
    if not cands: print("NO CANDIDATE for", fname, "- upload it manually (next cell)"); continue
    start = min(PICK.get(fname, 0), len(cands) - 1)
    ok = False
    for c in cands[start:start + 6]:                 # try up to 6 candidates until one really downloads
        for url in (c["dl"], c["direct"]):           # 1200px version first, then the original file
            for attempt in range(3):
                try:
                    resp = requests.get(url, headers=HDR, timeout=120)
                    if resp.status_code == 429: time.sleep(8 * (attempt + 1)); continue
                    resp.raise_for_status()
                    img = cv2.imdecode(np.frombuffer(resp.content, np.uint8), cv2.IMREAD_COLOR)
                    if img is None: raise ValueError("response is not an image")
                    cv2.imwrite(os.path.join(IMG_DIR, fname), img)
                    ok = True; break
                except Exception as e:
                    print("  retry", fname, "-", e); time.sleep(3)
            if ok: break
        if ok: break
        print("  candidate failed, trying next:", c["title"][:50])
    if not ok: print("DOWNLOAD FAILED", fname, "- re-run the cell or upload manually"); continue
    used.add(c["page"])
    records.append([fname, c["title"], c["desc"], c["page"], c["direct"], c["license"], c["author"], c["uploaded"]])
    print(f"OK {fname:24s} <- {c['title'][:60]}  [{c['license']}]")

hdr = ["filename", "commons_title", "description", "source_page", "direct_image_url", "license", "author", "uploaded"]
for d in (IMG_DIR, OUT_DIR):
    with open(os.path.join(d, "sources.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(hdr); w.writerows(records)

got = [r[0] for r in records]
fig, ax = plt.subplots(1, max(len(got), 2), figsize=(3.2 * max(len(got), 2), 3.4))
for a, n in zip(np.atleast_1d(ax), got):
    show(a, load(n, max_side=500), n)
plt.tight_layout(); plt.show()
print("\nSource, link and licence for each photo are in images/sources.csv (use them in the report).")

### Optional: replace one image by uploading your own
Skip this cell unless a downloaded photo looks wrong and `PICK` did not fix it.

In [ ]:
# from google.colab import files; import shutil
# up = files.upload()
# for fname in up:
#     new = input(f"Save '{fname}' as: ").strip() or fname
#     shutil.move(fname, os.path.join(IMG_DIR, new))

In [ ]:
need = ["outdoor.jpg","gray_photo.jpg","compress_photo.jpg","jpeg_image.jpg","textures.jpg","leaf.jpg"]
for n in need: print(("OK      " if os.path.exists(f"{IMG_DIR}/{n}") else "MISSING ") + n)

## Problem 1: Comparative edge detection

In [ ]:
color = load("outdoor.jpg", max_side=1000)
gray = cv2.cvtColor(color, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5, 5), 1.2)
gx = cv2.Sobel(blur, cv2.CV_64F, 1, 0, ksize=3); gy = cv2.Sobel(blur, cv2.CV_64F, 0, 1, ksize=3)
sobel = cv2.normalize(np.hypot(gx, gy), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
lap = cv2.convertScaleAbs(cv2.Laplacian(blur, cv2.CV_64F, ksize=3))
med = np.median(blur); canny = cv2.Canny(blur, int(0.66*med), int(1.33*med))
for name, e in [("Sobel", sobel), ("Laplacian", lap), ("Canny", canny)]:
    _, b = cv2.threshold(e, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    print(f"{name:10s} edge density = {100*np.mean(b>0):.2f}%")
fig, ax = plt.subplots(2, 2, figsize=(12, 9))
show(ax[0,0], color, "Original"); show(ax[0,1], sobel, "Sobel (magnitude)")
show(ax[1,0], lap, "Laplacian"); show(ax[1,1], canny, "Canny")
plt.tight_layout(); save(fig, "problem1_edges.png"); plt.show()

## Problem 2: Noise sensitivity

In [ ]:
rng = np.random.default_rng(42)
gray = load("gray_photo.jpg", gray=True, max_side=800)
gauss = np.clip(gray + rng.normal(0, 25, gray.shape), 0, 255).astype(np.uint8)
sp = gray.copy(); r = rng.random(gray.shape); sp[r < 0.025] = 0; sp[r > 0.975] = 255

def sobel(i):
    i = i.astype(np.float64)
    return cv2.normalize(np.hypot(cv2.Sobel(i, cv2.CV_64F, 1, 0), cv2.Sobel(i, cv2.CV_64F, 0, 1)),
                         None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
def lap(i): return cv2.convertScaleAbs(cv2.Laplacian(i, cv2.CV_64F, ksize=3))
def corr(a, b): return np.corrcoef(a.ravel().astype(float), b.ravel().astype(float))[0, 1]

ref_s, ref_l = sobel(gray), lap(gray)
cases = [("Gaussian noise", gauss, cv2.GaussianBlur(gauss, (5, 5), 1.4)),
         ("Salt & pepper", sp, cv2.GaussianBlur(cv2.medianBlur(sp, 3), (5, 5), 1.0))]
fig, ax = plt.subplots(3, 6, figsize=(18, 9))
for a in ax.ravel(): a.axis("off")
show(ax[0,0], gray, "Original"); show(ax[0,1], ref_s, "Sobel (clean)"); show(ax[0,2], ref_l, "Laplacian (clean)")
print(f"{'Case':16s}{'Stage':14s}{'Sobel corr':>11s}{'Lap corr':>10s}")
for row, (name, noisy, proc) in enumerate(cases, start=1):
    show(ax[row,0], noisy, name)
    show(ax[row,1], sobel(noisy), "Sobel (noisy)"); show(ax[row,2], lap(noisy), "Laplacian (noisy)")
    show(ax[row,3], proc, "Preprocessed")
    show(ax[row,4], sobel(proc), "Sobel (preproc.)"); show(ax[row,5], lap(proc), "Laplacian (preproc.)")
    print(f"{name:16s}{'noisy':14s}{corr(sobel(noisy), ref_s):11.3f}{corr(lap(noisy), ref_l):10.3f}")
    print(f"{name:16s}{'preprocessed':14s}{corr(sobel(proc), ref_s):11.3f}{corr(lap(proc), ref_l):10.3f}")
plt.tight_layout(); save(fig, "problem2_noise.png"); plt.show()

## Problem 3: RLE vs Huffman
Assumptions: each RLE pair costs 16 bits (8-bit value + 8-bit count, runs capped at 255); Huffman table counted as 256 x 8 bits. Negative reduction means expansion.

In [ ]:
def rle_encode(flat, max_run=255):
    change = np.flatnonzero(np.diff(flat)) + 1
    starts, ends = np.r_[0, change], np.r_[change, len(flat)]
    pairs = []
    for s, e in zip(starts, ends):
        v, L = int(flat[s]), int(e - s)
        while L > max_run: pairs.append((v, max_run)); L -= max_run
        pairs.append((v, L))
    return pairs
def rle_decode(pairs): return np.concatenate([np.full(c, v, np.uint8) for v, c in pairs])

def huffman_codes(freq):
    cnt = itertools.count()
    heap = [(f, next(cnt), {s: ""}) for s, f in freq.items()]; heapq.heapify(heap)
    if len(heap) == 1: return {s: "0" for s in heap[0][2]}
    while len(heap) > 1:
        f1, _, c1 = heapq.heappop(heap); f2, _, c2 = heapq.heappop(heap)
        m = {s: "0"+c for s, c in c1.items()}; m.update({s: "1"+c for s, c in c2.items()})
        heapq.heappush(heap, (f1+f2, next(cnt), m))
    return heap[0][2]

rows = []
for name in ["compress_photo.jpg", "compress_document.png"]:
    if not os.path.exists(os.path.join(IMG_DIR, name)): continue
    img = load(name, gray=True, max_side=512)
    flat = img.flatten(); N = flat.size; orig = N * 8
    pairs = rle_encode(flat); assert np.array_equal(rle_decode(pairs), flat), "RLE not lossless"
    rle_bits = len(pairs) * 16
    freq = Counter(flat.tolist()); codes = huffman_codes(freq)
    bits = "".join(codes[p] for p in flat.tolist())
    inv, out, cur = {c: s for s, c in codes.items()}, [], ""
    for b in bits:
        cur += b
        if cur in inv: out.append(inv[cur]); cur = ""
    assert out == flat.tolist(), "Huffman not lossless"
    payload, header = len(bits), 256 * 8
    p = np.array(list(freq.values())) / N; entropy = -(p * np.log2(p)).sum()
    for method, comp in [("RLE (8b value + 8b count)", rle_bits), ("Huffman (payload only)", payload),
                         ("Huffman (payload + 2048-bit table)", payload + header)]:
        rows.append([name, method, orig, comp, round(orig/comp, 3), round(100*(orig-comp)/orig, 2)])
    print(f"{name}: pixels={N}, runs={len(pairs)}, mean run={N/len(pairs):.2f}, entropy={entropy:.3f} bits/pixel")

hdr = ["Image", "Method", "Original bits", "Compressed bits", "Ratio", "Reduction %"]
print("\n" + " | ".join(hdr)); [print(" | ".join(map(str, r))) for r in rows]
with open(f"{OUT_DIR}/problem3_results.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(hdr); w.writerows(rows)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar([f"{r[0][:10]}\n{r[1][:12]}" for r in rows], [r[5] for r in rows])
ax.axhline(0, color="k", lw=0.8); ax.set_ylabel("Reduction % (negative = expansion)"); ax.set_title("RLE vs Huffman")
plt.tight_layout(); save(fig, "problem3_comparison.png"); plt.show()

## Problem 4: JPEG block artifacts (8x8 DCT, quantization, IDCT; PSNR and SSIM)

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr, structural_similarity as ssim
Q50 = np.array([[16,11,10,16,24,40,51,61],[12,12,14,19,26,58,60,55],[14,13,16,24,40,57,69,56],
                [14,17,22,29,51,87,80,62],[18,22,37,56,68,109,103,77],[24,35,55,64,81,104,113,92],
                [49,64,78,87,103,121,120,101],[72,92,95,98,112,100,103,99]], np.float32)
def scaled_q(quality):
    s = 5000 / quality if quality < 50 else 200 - 2 * quality
    return np.clip(np.floor((Q50 * s + 50) / 100), 1, 255).astype(np.float32)

def jpeg_like(gray, Q):
    h, w = gray.shape
    p = cv2.copyMakeBorder(gray, 0, (-h) % 8, 0, (-w) % 8, cv2.BORDER_REPLICATE).astype(np.float32) - 128
    out, nonzero = np.zeros_like(p), 0
    for i in range(0, p.shape[0], 8):
        for j in range(0, p.shape[1], 8):
            q = np.round(cv2.dct(p[i:i+8, j:j+8]) / Q)
            nonzero += np.count_nonzero(q); out[i:i+8, j:j+8] = cv2.idct(q * Q)
    return np.clip(np.round(out + 128), 0, 255).astype(np.uint8)[:h, :w], 100 * nonzero / p.size

gray = load("jpeg_image.jpg", gray=True, max_side=800)
h, w = gray.shape; y0, x0 = h//2 - 32, w//2 - 32
def zoom(i): return cv2.resize(i[y0:y0+64, x0:x0+64], None, fx=4, fy=4, interpolation=cv2.INTER_NEAREST)
levels = {"Low (Q=90)": 90, "Medium (Q=50)": 50, "High (Q=10)": 10}
fig, ax = plt.subplots(2, 4, figsize=(18, 9))
show(ax[0,0], gray, "Original"); show(ax[1,0], zoom(gray), "Original crop x4")
rows = []
for k, (name, q) in enumerate(levels.items(), start=1):
    rec, nz = jpeg_like(gray, scaled_q(q))
    P, S = psnr(gray, rec, data_range=255), ssim(gray, rec, data_range=255)
    rows.append([name, round(P, 2), round(S, 4), round(nz, 2)])
    show(ax[0,k], rec, f"{name}\nPSNR={P:.2f} dB SSIM={S:.3f}"); show(ax[1,k], zoom(rec), "crop x4")
hdr = ["Level", "PSNR (dB)", "SSIM", "Non-zero coeffs (%)"]
print(" | ".join(hdr)); [print(" | ".join(map(str, r))) for r in rows]
with open(f"{OUT_DIR}/problem4_results.csv", "w", newline="") as f:
    wr = csv.writer(f); wr.writerow(hdr); wr.writerows(rows)
plt.tight_layout(); save(fig, "problem4_jpeg.png"); plt.show()

## Problem 5: GLCM texture segmentation (may take about a minute; raise STEP to 6 if slow)

In [ ]:
from skimage.feature import graycomatrix, graycoprops
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

gray = load("textures.jpg", gray=True, max_side=400)
H, W = gray.shape
WIN, STEP, LEVELS = 15, 4, 32
q = (gray // (256 // LEVELS)).astype(np.uint8)
pad = WIN // 2
qp = cv2.copyMakeBorder(q, pad, pad, pad, pad, cv2.BORDER_REFLECT)
ys, xs = range(0, H, STEP), range(0, W, STEP)
feats = np.zeros((len(ys), len(xs), 4), np.float32)   # contrast, energy, homogeneity, mean intensity
for a, y in enumerate(ys):
    for b, x in enumerate(xs):
        patch = qp[y:y+WIN, x:x+WIN]
        g = graycomatrix(patch, [1], [0, np.pi/2], levels=LEVELS, symmetric=True, normed=True)
        feats[a, b] = [graycoprops(g, p).mean() for p in ("contrast","energy","homogeneity")] + [patch.mean()]

def segment(cols):
    X = StandardScaler().fit_transform(feats.reshape(-1, 4)[:, cols])
    km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
    sil = silhouette_score(X, km.labels_, sample_size=min(3000, len(X)), random_state=0)
    lab = km.labels_.reshape(feats.shape[:2]).astype(np.uint8)
    return cv2.resize(lab, (W, H), interpolation=cv2.INTER_NEAREST), sil

exps = {"A: intensity only": [3], "B: GLCM only (C,E,H)": [0,1,2], "C: GLCM + intensity": [0,1,2,3]}
big = lambda i: cv2.resize(feats[..., i], (W, H), interpolation=cv2.INTER_LINEAR)
fig, ax = plt.subplots(2, 4, figsize=(18, 8))
show(ax[0,0], gray, "Original")
for i, t in enumerate(["GLCM contrast","GLCM energy","GLCM homogeneity"], start=1): show(ax[0,i], big(i-1), t, "viridis")
rows = []
for k, (name, cols) in enumerate(exps.items()):
    seg, sil = segment(cols); rows.append([name, round(float(sil), 3)])
    show(ax[1,k], seg, f"{name}\nsilhouette={sil:.3f}", "viridis")
show(ax[1,3], big(3), "Mean intensity map", "gray")
print("Experiment | Silhouette"); [print(" | ".join(map(str, r))) for r in rows]
with open(f"{OUT_DIR}/problem5_results.csv", "w", newline="") as f:
    wr = csv.writer(f); wr.writerow(["Experiment","Silhouette"]); wr.writerows(rows)
plt.tight_layout(); save(fig, "problem5_texture.png"); plt.show()

## Problem 6: Multi-feature leaf segmentation
Unsupervised, heuristic segmentation. It is NOT a medically or scientifically validated diagnosis. Shadows, veins, background and leaf edges may be wrongly grouped as "diseased".

In [ ]:
color = load("leaf.jpg", max_side=600)
gray = cv2.cvtColor(color, cv2.COLOR_BGR2GRAY).astype(np.float32)
H, W = gray.shape
gb = cv2.GaussianBlur(gray, (3, 3), 0)
mag = np.hypot(cv2.Sobel(gb, cv2.CV_32F, 1, 0, ksize=3), cv2.Sobel(gb, cv2.CV_32F, 0, 1, ksize=3))
m = cv2.boxFilter(gray, -1, (7, 7)); var = np.maximum(cv2.boxFilter(gray*gray, -1, (7, 7)) - m*m, 0)

X = StandardScaler().fit_transform(np.column_stack([gray.ravel(), np.log1p(mag.ravel()), np.log1p(var.ravel())]))
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
lab = km.labels_.reshape(H, W)
order = np.argsort([gray[lab == c].mean() for c in range(3)])
remap = {int(old): new for new, old in enumerate(order)}
lab = np.vectorize(remap.get)(lab)
rows = [[c, round(100*np.mean(lab == c), 2), round(float(gray[lab == c].mean()), 1),
         round(float(mag[lab == c].mean()), 1), round(float(var[lab == c].mean()), 1)] for c in range(3)]
hdr = ["Cluster (0=darkest)", "Area %", "Mean intensity", "Mean Sobel", "Mean local variance"]
print(" | ".join(hdr)); [print(" | ".join(map(str, r))) for r in rows]
with open(f"{OUT_DIR}/problem6_results.csv", "w", newline="") as f:
    wr = csv.writer(f); wr.writerow(hdr); wr.writerows(rows)

norm = lambda a: cv2.normalize(a, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
overlay = color.copy(); overlay[lab == 0] = (0, 0, 255)
overlay = cv2.addWeighted(color, 0.5, overlay, 0.5, 0)
fig, ax = plt.subplots(2, 3, figsize=(16, 9))
show(ax[0,0], color, "Original photograph"); show(ax[0,1], norm(gray), "Feature 1: intensity")
show(ax[0,2], norm(mag), "Feature 2: Sobel magnitude"); show(ax[1,0], norm(var), "Feature 3: local variance")
show(ax[1,1], lab, "K-means (k=3) segmentation", "nipy_spectral")
show(ax[1,2], overlay, "Darkest cluster (red) = candidate lesion\nHeuristic only, NOT a diagnosis")
plt.tight_layout(); save(fig, "problem6_leaf.png"); plt.show()

## Problem 7 : Contrast enhancement, histogram equalization vs CLAHE
Aim: improve the contrast of a real photograph and measure the change. Global histogram equalization spreads grey levels over the full range; CLAHE equalizes small tiles with a clip limit so noise is not over-amplified. Metrics: mean, standard deviation (contrast) and Shannon entropy (information).

In [ ]:
gray = load("gray_photo.jpg", gray=True, max_side=800)
he = cv2.equalizeHist(gray)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
def entropy(i):
    p = np.bincount(i.ravel(), minlength=256) / i.size; p = p[p > 0]
    return -(p * np.log2(p)).sum()
items = [("Original", gray), ("Histogram equalization", he), ("CLAHE (clip=2, 8x8)", clahe)]
rows = [[n, round(float(i.mean()), 2), round(float(i.std()), 2), round(float(entropy(i)), 3)] for n, i in items]
hdr = ["Image", "Mean", "Std (contrast)", "Entropy (bits)"]
print(" | ".join(hdr)); [print(" | ".join(map(str, r))) for r in rows]
with open(f"{OUT_DIR}/problem7_results.csv", "w", newline="") as f:
    wr = csv.writer(f); wr.writerow(hdr); wr.writerows(rows)
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
for k, (n, i) in enumerate(items):
    show(ax[0, k], i, n)
    ax[1, k].hist(i.ravel(), bins=256, range=(0, 255)); ax[1, k].set_title("Histogram: " + n, fontsize=9)
plt.tight_layout(); save(fig, "problem7_contrast.png"); plt.show()

## Problem 8: Denoising filter comparison (mean, Gaussian, median, bilateral)
Aim: find which spatial filter removes Gaussian noise and salt-and-pepper noise best. PSNR and SSIM are measured against the clean photograph, so higher is better.

In [ ]:
rng = np.random.default_rng(7)
clean = load("gray_photo.jpg", gray=True, max_side=600)
gauss_n = np.clip(clean + rng.normal(0, 25, clean.shape), 0, 255).astype(np.uint8)
sp_n = clean.copy(); r = rng.random(clean.shape); sp_n[r < 0.025] = 0; sp_n[r > 0.975] = 255
filters = {"Mean 5x5": lambda x: cv2.blur(x, (5, 5)),
           "Gaussian 5x5": lambda x: cv2.GaussianBlur(x, (5, 5), 1.4),
           "Median 5x5": lambda x: cv2.medianBlur(x, 5),
           "Bilateral": lambda x: cv2.bilateralFilter(x, 9, 75, 75)}
rows = []
fig, ax = plt.subplots(2, 5, figsize=(20, 8))
for row, (nname, noisy) in enumerate([("Gaussian noise", gauss_n), ("Salt & pepper", sp_n)]):
    rows.append([nname, "No filter", round(psnr(clean, noisy, data_range=255), 2), round(ssim(clean, noisy, data_range=255), 4)])
    show(ax[row, 0], noisy, f"{nname}\nPSNR={rows[-1][2]} SSIM={rows[-1][3]}")
    for col, (fname, fn) in enumerate(filters.items(), start=1):
        out = fn(noisy)
        P, S = psnr(clean, out, data_range=255), ssim(clean, out, data_range=255)
        rows.append([nname, fname, round(P, 2), round(S, 4)])
        show(ax[row, col], out, f"{fname}\nPSNR={P:.2f} SSIM={S:.3f}")
hdr = ["Noise", "Filter", "PSNR (dB)", "SSIM"]
print(" | ".join(hdr)); [print(" | ".join(map(str, r))) for r in rows]
with open(f"{OUT_DIR}/problem8_results.csv", "w", newline="") as f:
    wr = csv.writer(f); wr.writerow(hdr); wr.writerows(rows)
plt.tight_layout(); save(fig, "problem8_denoise.png"); plt.show()

## Problem 9 : Thresholding segmentation (global, Otsu, adaptive)
Aim: separate foreground from background using a fixed threshold, Otsu's automatic threshold, and local adaptive thresholds. Uses the document image if it was downloaded, otherwise the grayscale photo.

In [ ]:
name9 = "compress_document.png" if os.path.exists(f"{IMG_DIR}/compress_document.png") else "gray_photo.jpg"
img = load(name9, gray=True, max_side=900)
blur = cv2.GaussianBlur(img, (5, 5), 0)
t_otsu, otsu = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
_, fixed = cv2.threshold(blur, 127, 255, cv2.THRESH_BINARY)
ad_mean = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 31, 10)
ad_gauss = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 10)
res = [("Global T=127", fixed), (f"Otsu T={t_otsu:.0f}", otsu), ("Adaptive mean", ad_mean), ("Adaptive Gaussian", ad_gauss)]
rows = [[n, round(100 * np.mean(b == 255), 2)] for n, b in res]
print("Image used:", name9)
print("Method | White pixels (%)"); [print(" | ".join(map(str, r))) for r in rows]
with open(f"{OUT_DIR}/problem9_results.csv", "w", newline="") as f:
    wr = csv.writer(f); wr.writerow(["Method", "White pixels %"]); wr.writerows(rows)
fig, ax = plt.subplots(2, 3, figsize=(16, 9))
show(ax[0, 0], img, f"Original ({name9})")
ax[0, 1].hist(img.ravel(), bins=256, range=(0, 255)); ax[0, 1].axvline(t_otsu, color="r", label="Otsu"); ax[0, 1].axvline(127, color="g", label="127"); ax[0, 1].legend(); ax[0, 1].set_title("Histogram", fontsize=9)
ax[0, 2].axis("off")
show(ax[0, 2], fixed, "Global T=127"); show(ax[1, 0], otsu, f"Otsu T={t_otsu:.0f}")
show(ax[1, 1], ad_mean, "Adaptive mean (31, C=10)"); show(ax[1, 2], ad_gauss, "Adaptive Gaussian (31, C=10)")
plt.tight_layout(); save(fig, "problem9_threshold.png"); plt.show()

## Problem 10 : Frequency-domain filtering with the 2D FFT
Aim: apply Gaussian low-pass and high-pass filters in the Fourier domain. Low-pass keeps smooth content (blur); high-pass keeps edges and fine detail. Energy retained is the share of spectrum energy kept by the filter.

In [ ]:
gray = load("gray_photo.jpg", gray=True, max_side=512)
Hh, Ww = gray.shape
F = np.fft.fftshift(np.fft.fft2(gray.astype(np.float64)))
u, v = np.meshgrid(np.arange(Ww) - Ww // 2, np.arange(Hh) - Hh // 2)
D = np.sqrt(u**2 + v**2)
def gauss_lp(d0): return np.exp(-(D**2) / (2 * d0**2))
def apply(Hmask): return np.real(np.fft.ifft2(np.fft.ifftshift(F * Hmask)))
def energy(Hmask): return 100 * np.sum(np.abs(F * Hmask)**2) / np.sum(np.abs(F)**2)
spec = np.log1p(np.abs(F))
rows, outs = [], []
for label, Hmask, is_hp in [("Low-pass D0=20", gauss_lp(20), False), ("Low-pass D0=60", gauss_lp(60), False),
                            ("High-pass D0=30", 1 - gauss_lp(30), True), ("High-pass D0=60", 1 - gauss_lp(60), True)]:
    rec = apply(Hmask)
    if is_hp:
        disp = cv2.normalize(np.abs(rec), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8); P = S = float("nan")
    else:
        disp = np.clip(rec, 0, 255).astype(np.uint8)
        P, S = psnr(gray, disp, data_range=255), ssim(gray, disp, data_range=255)
    rows.append([label, round(energy(Hmask), 2), round(P, 2), round(S, 4)]); outs.append((label, disp))
hdr = ["Filter", "Energy retained (%)", "PSNR vs original (dB)", "SSIM vs original"]
print(" | ".join(hdr)); [print(" | ".join(map(str, r))) for r in rows]
with open(f"{OUT_DIR}/problem10_results.csv", "w", newline="") as f:
    wr = csv.writer(f); wr.writerow(hdr); wr.writerows(rows)
fig, ax = plt.subplots(2, 3, figsize=(16, 9))
show(ax[0, 0], gray, "Original"); show(ax[0, 1], cv2.normalize(spec, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8), "Log magnitude spectrum (centred)")
show(ax[0, 2], outs[0][1], outs[0][0]); show(ax[1, 0], outs[1][1], outs[1][0])
show(ax[1, 1], outs[2][1], outs[2][0] + " (abs, normalised)"); show(ax[1, 2], outs[3][1], outs[3][0] + " (abs, normalised)")
plt.tight_layout(); save(fig, "problem10_fft.png"); plt.show()

## Download all results

In [ ]:
import shutil
zip_path = shutil.make_archive("lab_outputs", "zip", OUT_DIR)
print("Saved:", os.path.abspath(zip_path))
try:                                   # works only inside Google Colab
    from google.colab import files
    files.download(zip_path)
except ImportError:                    # Jupyter: the zip and the outputs/ folder are in your working folder
    print("Jupyter detected: open the folder shown above to find lab_outputs.zip and outputs/")